# Introduction

## Model Building and Evaluation

This notebook focuses on:
- Training multiple machine learning models on the Wisconsin Breast Cancer Dataset.
- Comparing model performance on both the original and PCA-reduced datasets.
- Performing hyperparameter tuning for each model to achieve the best results.

Models Used:
- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine (SVM)

# Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Loading the data

In [2]:
columns = [
    "ID", "Diagnosis", "Radius_mean", "Texture_mean", "Perimeter_mean", "Area_mean", "Smoothness_mean", 
    "Compactness_mean", "Concavity_mean", "Concave_points_mean", "Symmetry_mean", "Fractal_dimension_mean",
    "Radius_se", "Texture_se", "Perimeter_se", "Area_se", "Smoothness_se", "Compactness_se", "Concavity_se",
    "Concave_points_se", "Symmetry_se", "Fractal_dimension_se", "Radius_worst", "Texture_worst", "Perimeter_worst",
    "Area_worst", "Smoothness_worst", "Compactness_worst", "Concavity_worst", "Concave_points_worst",
    "Symmetry_worst", "Fractal_dimension_worst"
]
data = pd.read_csv("wdbc.data",header=None, names=columns)
data = data.drop("ID", axis=1) 
data["Diagnosis"] = data["Diagnosis"].map({"B": 0, "M": 1})
X = data.drop('Diagnosis', axis=1)
y = data['Diagnosis']
pca = PCA(n_components=3)  
X_pca = pca.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=1, stratify=y)

# Model Training and Testing

## Model Training and Testing

The following models are trained and evaluated:
- **Logistic Regression**
- **Decision Tree**
- **Random Forest**
- **SVM**

For each model:
- The original dataset and PCA-reduced dataset are used.
- **Cross-validation** is applied with 5 folds to calculate the mean accuracy.

In [3]:
#Initialize classifiers
classifiers = [
    ("Logistic Regression", LogisticRegression(random_state = 1)),
    ("Decision Tree", DecisionTreeClassifier(random_state = 1)),
    ("Random Forest", RandomForestClassifier(random_state = 1)),
    ("SVM", SVC(random_state = 1))
]

#Apply classifiers to both original and PCA datasets
for name, clf in classifiers:
    #Original dataset
    results = cross_validate(clf, X_train, y_train, cv=5, scoring = 'accuracy')
    print(f"\n{name} Accuracy: {results['test_score'].mean()} ")
    #PCA dataset
    results = cross_validate(clf, X_train_pca, y_train_pca, cv=5, scoring = 'accuracy')
    print(f"\n{name} with PCA Accuracy: {results['test_score'].mean()}")


Logistic Regression Accuracy: 0.9472527472527472 

Logistic Regression with PCA Accuracy: 0.9274725274725274

Decision Tree Accuracy: 0.9406593406593406 

Decision Tree with PCA Accuracy: 0.8747252747252748

Random Forest Accuracy: 0.9582417582417582 

Random Forest with PCA Accuracy: 0.9208791208791208

SVM Accuracy: 0.9054945054945055 

SVM with PCA Accuracy: 0.9164835164835164


- Random Forest and Logistic Regression have good performance.
- PCA reduced the dataset dimensionality while retaining most of the variance.
- In most of the models, PCA-applied dataset showed lower accuracy as some of the information is lost after dimenionality reduction.

# Tuning Hyperparamters

## Hyperparameter Tuning

GridSearchCV is used to find the optimal hyperparameters for each model.

## Hyperparameters Tuned:
- **Logistic Regression**:
  - `C`: Regularization strength.
  - `solver`: Optimization algorithm.
  - `penalty`: Regularization type.
- **Decision Tree**:
  - `max_depth`: Maximum depth of the tree.
  - `min_samples_split`: Minimum samples required to split a node.
  - `min_samples_leaf`: Minimum samples required in a leaf node.
- **Random Forest**:
  - `n_estimators`: Number of trees.
  - `max_depth`: Maximum depth of each tree.
  - `min_samples_split`: Minimum samples to split.
- **SVM**:
  - `C`: Regularization parameter.
  - `kernel`: Kernel type (linear, RBF).

## Model Evaluation

Evaluation metrics used:
- **Accuracy**: Percentage of correct predictions.
- **Precision**: Proportion of true positive predictions among all positive predictions.
- **Recall**: Proportion of true positives identified.
- **F1-Score**: Harmonic mean of precision and recall.

In [4]:
# Function to evaluate models
def evaluate_model(y_test, y_pred):
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [5]:
print("\nLogistic Regression:")
param_grid_lr = {"C": [0.1, 1, 10, 100], "solver": ["liblinear", "saga"], "penalty":["l1","l2"],"max_iter": [100, 500, 1000]}
log_reg = GridSearchCV(LogisticRegression(random_state = 1), param_grid_lr, cv=5)
log_reg.fit(X_train, y_train)
print(f"Best Params: {log_reg.best_params_}")
y_pred = log_reg.predict(X_test)
evaluate_model(y_test,y_pred)


Logistic Regression:
Best Params: {'C': 100, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'}
Accuracy: 0.9825
Precision: 0.9762
Recall: 0.9762
F1 Score: 0.9762
Confusion Matrix:
[[71  1]
 [ 1 41]]


In [6]:
print("\nDecision Tree:")
param_grid_dt = {"max_depth": [3, 5, 10, 20, None], "min_samples_split": [2, 5, 10], "min_samples_leaf": [2, 5, 10]}
decision_tree = GridSearchCV(DecisionTreeClassifier(random_state = 1), param_grid_dt, cv=5)
decision_tree.fit(X_train, y_train)
print(f"Best Params: {decision_tree.best_params_}")
y_pred = decision_tree.predict(X_test)
evaluate_model(y_test,y_pred)


Decision Tree:
Best Params: {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 2}
Accuracy: 0.9211
Precision: 0.8837
Recall: 0.9048
F1 Score: 0.8941
Confusion Matrix:
[[67  5]
 [ 4 38]]


In [7]:
print("\nRandom Forest:")
param_grid_rf = {"n_estimators": [100, 200], "max_depth": [5, 10, None], "min_samples_split": [2, 5, 10]}
random_forest = GridSearchCV(RandomForestClassifier(random_state = 1), param_grid_rf, cv=5)
random_forest.fit(X_train, y_train)
print(f"Best Params: {random_forest.best_params_}")
y_pred = random_forest.predict(X_test)
evaluate_model(y_test,y_pred)


Random Forest:
Best Params: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 200}
Accuracy: 0.9474
Precision: 0.9737
Recall: 0.8810
F1 Score: 0.9250
Confusion Matrix:
[[71  1]
 [ 5 37]]


In [8]:
print("\nSVM:")
param_grid_svm = {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]}
svm = GridSearchCV(SVC(random_state = 1), param_grid_svm, cv=5)
svm.fit(X_train, y_train)
print(f"Best Params: {svm.best_params_}")
y_pred = svm.predict(X_test)
evaluate_model(y_test,y_pred)


SVM:
Best Params: {'C': 0.1, 'kernel': 'linear'}
Accuracy: 0.9561
Precision: 0.9744
Recall: 0.9048
F1 Score: 0.9383
Confusion Matrix:
[[71  1]
 [ 4 38]]


## Conclusion

- **Best Model**: Logistic Regression
- **Best Accuracy**: 98.25%